In [5]:
import warnings
import os
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import pandas as pd
import numpy as np
import random
import sc_toolbox

import rpy2.rinterface_lib.callbacks
import anndata2ri
import logging
from pathlib import Path
from rpy2.robjects import pandas2ri, numpy2ri
from rpy2.robjects import r
from scipy.sparse import csr_matrix
import anndata as ad
sc.settings.verbosity = 0
rpy2.rinterface_lib.callbacks.logger.setLevel(logging.ERROR)

pandas2ri.activate()
numpy2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [6]:

%%R
library(edgeR)
library(MAST)

In [7]:
project_path = Path(
    "/Users/nicolebussola/Desktop/Projects/LBP_MountSinai/LBP_brain_blood_pairs"
)
adata = sc.read(project_path / "data" / "blood_4000chem_scvi_clustered_annotated.h5ad")

In [25]:
meta = pd.read_csv("../../LBP_brain_blood_pairs/data/narsad_cellRanger_outs/Batch1_celllevel_metadata.tsv", sep="\t")
from collections import Counter
Counter(meta['donor_side']),Counter(meta['Working.Dx']),Counter(meta[ 'age']), Counter(meta['sex'])
meta.groupby(['donor_side','age'])[ 'age'].count()

donor_side  age
167_L       67      8989
182_R       66      8596
185_L       70     12301
201_L       68      8275
203_L       55      5084
203_R       55      9136
205_L       72      5725
205_R       72      9220
206_L       69      6473
208_L       63      5608
208_R       63      4049
212_L       68      5851
212_R       68      6440
214_L       21      4567
214_R       21      7003
Name: age, dtype: int64

In [22]:
adata.obs['manual_celltype_annotation_bp_175'].unique()
adata[adata.obs['manual_celltype_annotation_bp_175'].isin(['Garbage', 'NA'])]

View of AnnData object with n_obs × n_vars = 2049 × 4000
    obs: 'n_genes_x', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'scDblFinder_score', 'scDblFinder_class', 'doublet_scores_scrublet', 'predicted_doublets_scrublet', 'cluster_labels', 'n_counts_lower_co', 'n_counts_upper_co', 'n_counts', 'n_counts_passed_qc', 'n_genes_lower_co', 'n_genes_upper_co', 'n_genes_y', 'n_genes_passed_qc', 'percent_mito_lower_co', 'percent_mito_upper_co', 'percent_mito', 'percent_mito_passed_qc', 'percent_ribo_lower_co', 'percent_ribo_upper_co', 'percent_ribo', 'percent_ribo_passed_qc', 'passed_qc', 'side', 'pt', 'chemistry', 'scrublet_class_refined', 'size_factors', '_scvi_batch', '_scvi_labels', 'leiden_res0_25', 'leiden_res0_5', 'leid

In [9]:
adata.X = adata.layers["counts"].copy()


In [11]:
np.max(adata.X), np.max(adata.X)
adata.obs.head()

,n_genes_x,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_20_genes,total_counts_mt,log1p_total_counts_mt,pct_counts_mt,total_counts_ribo,...,leiden_res1,leiden_res1_25,leiden_res1_5,leiden_res1_75,celltypist_cell_label_coarse,celltypist_conf_score_coarse,celltypist_cell_label_fine,celltypist_conf_score_fine,manual_celltype_annotation_bp_15,manual_celltype_annotation_bp_175
GATTCAGGTCCAACTA_0,3698,3697,8.215547,15718.0,9.662625,18.373839,653.0,6.483108,4.154472,4086.0,...,13,17,18,23,DC,0.223811,DC2,0.105452,cDC2,cDC2
CTCTACGTCAGGTAAA_0,2950,2949,7.989560,15273.0,9.633907,25.063838,343.0,5.840641,2.245793,4988.0,...,13,17,18,31,DC,0.999221,DC1,0.977049,cDC2,cDC1
GTGGGTCTCCAGATCA_0,3373,3372,8.123558,14625.0,9.590556,20.205128,500.0,6.216606,3.418804,3281.0,...,13,17,18,31,DC,0.991528,DC1,0.884800,cDC2,cDC1
TTGTAGGAGCCCGAAA_0,3153,3152,8.056110,13792.0,9.531917,18.445476,343.0,5.840641,2.486949,4530.0,...,4,3,14,11,Monocytes,0.560219,Classical monocytes,0.548694,CD14+ Mono,CD14+ Mono
ACGGGTCCATGTTCCC_0,2812,2811,7.941651,12123.0,9.402943,19.788831,410.0,6.018593,3.382001,3112.0,...,8,7,8,10,Monocytes,0.999980,Classical monocytes,0.986661,CD14+ Mono,CD14+ Mono


In [26]:
adata.obs["pt"] = [pt.replace("-", "_") for pt in adata.obs["pt"]]
adata.obs['cell_type'] = adata.obs['manual_celltype_annotation_bp_175']
adata.obs["cell_type"] = [ct.replace(" ", "_") for ct in adata.obs["cell_type"]]
adata.obs["cell_type"] = [ct.replace("+", "") for ct in adata.obs["cell_type"]]


In [27]:
adata.obs["sample"] = [
    f"{rep}_{l}" for rep, l in zip(adata.obs["pt"], adata.obs["side"])
]

adata.obs["cell_type"] = adata.obs["cell_type"].astype("category")
adata.obs["sample"] = adata.obs["sample"].astype("category")
adata.obs["label"] = adata.obs["side"].astype("category")
adata.obs["replicate"] = adata.obs["pt"].astype("category")
print(pd.unique(adata.obs["cell_type"]),pd.unique(adata.obs["sample"]))

['cDC2', 'cDC1', 'CD14_Mono', 'CD4_T_Naive', 'CD16_Mono', ..., 'NA', 'Plasma_cells', 'MAIT_cells', 'Garbage', 'HSC_or_mast_cells?']
Length: 16
Categories (16, object): ['B_cells', 'CD14_Mono', 'CD16_Mono', 'CD4_T_Naive', ..., 'Plasma_cells', 'Platelets', 'cDC1', 'cDC2'] ['PT_182_R', 'PT_185_L', 'PT_201_L', 'PT_203_L', 'PT_203_R', ..., 'PT_208_R', 'PT_212_L', 'PT_212_R', 'PT_214_L', 'PT_214_R']
Length: 14
Categories (14, object): ['PT_182_R', 'PT_185_L', 'PT_201_L', 'PT_203_L', ..., 'PT_212_L', 'PT_212_R', 'PT_214_L', 'PT_214_R']


In [40]:
print(adata.shape)
indices = list(adata.obs_names)
random.shuffle(indices)
indices = np.array_split(np.array(indices), 1)
for i, rep_idx in enumerate(indices):
    print(adata[rep_idx].shape)


(38732, 4000)
(38732, 4000)


In [28]:
NUM_OF_CELL_PER_DONOR = 10

np.random.seed(42)

def aggregate_and_filter(
    adata,
    cell_identity,
    donor_key="sample",
    condition_key="label",
    cell_identity_key="cell_type",
    obs_to_keep=[],  # which additional metadata to keep, e.g. gender, age, etc.
    replicates_per_patient=1,
):
    # subset adata to the given cell identity
    adata_cell_pop = adata[adata.obs[cell_identity_key] == cell_identity].copy()
    # check which donors to keep according to the number of cells specified with NUM_OF_CELL_PER_DONOR
    size_by_donor = adata_cell_pop.obs.groupby([donor_key]).size()
    donors_to_drop = [
        donor
        for donor in size_by_donor.index
        if size_by_donor[donor] <= NUM_OF_CELL_PER_DONOR
    ]
    if len(donors_to_drop) > 0:
        print("Dropping the following samples:")
        print(donors_to_drop)
    df = pd.DataFrame(columns=[*adata_cell_pop.var_names, *obs_to_keep])

    adata_cell_pop.obs[donor_key] = adata_cell_pop.obs[donor_key].astype("category")
    for i, donor in enumerate(donors := adata_cell_pop.obs[donor_key].cat.categories):
        print(f"\tProcessing donor {i+1} out of {len(donors)}...", end="\r")
        if donor not in donors_to_drop:
            adata_donor = adata_cell_pop[adata_cell_pop.obs[donor_key] == donor]
            # create replicates for each donor
            indices = list(adata_donor.obs_names)
            random.shuffle(indices)
            indices = np.array_split(np.array(indices), replicates_per_patient)
            for i, rep_idx in enumerate(indices):
                adata_replicate = adata_donor[rep_idx]
                # specify how to aggregate: sum gene expression for each gene for each donor and also keep the condition information
                agg_dict = {gene: "sum" for gene in adata_replicate.var_names}
                for obs in obs_to_keep:
                    agg_dict[obs] = "first"
                # create a df with all genes, donor and condition info
                df_donor = pd.DataFrame(adata_replicate.X.A)
                df_donor.index = adata_replicate.obs_names
                df_donor.columns = adata_replicate.var_names
                df_donor = df_donor.join(adata_replicate.obs[obs_to_keep])
                # aggregate
                df_donor = df_donor.groupby(donor_key).agg(agg_dict)
                df_donor[donor_key] = donor
                df.loc[f"donor_{donor}_{i}"] = df_donor.loc[donor]
    print("\n")
    # create AnnData object from the df
    adata_cell_pop = sc.AnnData(
        df[adata_cell_pop.var_names], obs=df.drop(columns=adata_cell_pop.var_names)
    )
    return adata_cell_pop

In [29]:
%%R
fit_model <- function(adata_){
    # create an edgeR object with counts and grouping factor
    y <- DGEList(assay(adata_, "X"), group = colData(adata_)$label)
    # filter out genes with low counts
    print("Dimensions before subsetting:")
    print(dim(y))
    print("")
    keep <- filterByExpr(y)
    y <- y[keep, , keep.lib.sizes=FALSE]
    print("Dimensions after subsetting:")
    print(dim(y))
    print("")
    # normalize
    y <- calcNormFactors(y)
    # create a vector that is concatentation of condition and cell type that we will later use with contrasts
    group <- paste0(colData(adata_)$label, ".", colData(adata_)$cell_type)
    replicate <- colData(adata_)$replicate
    # create a design matrix: here we have multiple donors so also consider that in the design matrix
    design <- model.matrix(~ 0 + group + replicate)
    # estimate dispersion
    y <- estimateDisp(y, design = design)
    # fit the model
    fit <- glmQLFit(y, design)
    return(list("fit"=fit, "design"=design, "y"=y))
}

In [30]:
obs_to_keep = ["label", "cell_type", "replicate", "sample"]


In [31]:
# process first cell type separately...
cell_type = adata.obs["cell_type"].cat.categories[0]
print(
    f'Processing {cell_type} (1 out of {len(adata.obs["cell_type"].cat.categories)})...'
)
adata_pb = aggregate_and_filter(adata, cell_type,  donor_key="sample", condition_key="label", cell_identity_key="cell_type", obs_to_keep=obs_to_keep)
for i, cell_type in enumerate(adata.obs["cell_type"].cat.categories[1:]):
    print(
        f'Processing {cell_type} ({i+2} out of {len(adata.obs["cell_type"].cat.categories)})...'
    )
    adata_cell_type = aggregate_and_filter(adata, cell_type, obs_to_keep=obs_to_keep)
    adata_pb = adata_pb.concatenate(adata_cell_type)

Processing B_cells (1 out of 16)...
Dropping the following samples:
['PT_212_L']
	Processing donor 13 out of 13...

Processing CD14_Mono (2 out of 16)...
	Processing donor 14 out of 14...

Processing CD16_Mono (3 out of 16)...
Dropping the following samples:
['PT_208_L']
	Processing donor 14 out of 14...

Processing CD4_T_Naive (4 out of 16)...
	Processing donor 13 out of 13...

Processing CD8_T (5 out of 16)...
	Processing donor 14 out of 14...

Processing Cytotoxic_T_cells (6 out of 16)...
Dropping the following samples:
['PT_201_L', 'PT_208_L']
	Processing donor 11 out of 11...

Processing Garbage (7 out of 16)...
Dropping the following samples:
['PT_185_L', 'PT_201_L', 'PT_203_R', 'PT_205_L', 'PT_214_L']
	Processing donor 10 out of 10...

Processing HSC_or_mast_cells? (8 out of 16)...
Dropping the following samples:
['PT_185_L', 'PT_205_L', 'PT_208_L', 'PT_214_L', 'PT_214_R']
	Processing donor 6 out of 6...

Processing MAIT_cells (9 out of 16)...
Dropping the following samples:
['P

In [ ]:

adata_pb.obs

In [ ]:
adata_pb.layers['counts'] = adata_pb.X.copy()
sc.pp.normalize_total(adata_pb, target_sum=1e6)
sc.pp.log1p(adata_pb)
sc.pp.pca(adata_pb)

In [ ]:
adata_pb.obs["lib_size"] = np.sum(adata_pb.layers["counts"], axis=1)
# adata_pb.obs["log_lib_size"] = np.log(adata_pb.obs["lib_size"])

In [ ]:
sc.pl.pca(adata_pb, color=adata_pb.obs, ncols=1, size=300)


In [ ]:
adata_pb.layers['counts'] = csr_matrix(adata_pb.layers["counts"].astype(np.float32))
adata_pb.X = adata_pb.layers['counts'].copy()
type(adata_pb.X)

In [ ]:
adata_mono = adata_pb[adata_pb.obs["cell_type"] == "CD14_Mono"]
adata_mono

In [ ]:
adata_mono.obs_names = [
    name.split("_")[1] + "_" + name.split("_")[2] for name in adata_mono.obs_names
]

In [ ]:
adata_mono.obs['lib_size']=adata_mono.obs['lib_size'].astype(int)
adata_mono.obs['lib_size']

In [ ]:
%%time
%%R -i adata_mono
outs <-fit_model(adata_mono)

In [ ]:
%%R
fit <- outs$fit
y <- outs$y

In [ ]:
%%R
plotMDS(y, col=ifelse(y$samples$group == "stim", "red", "blue"))

In [ ]:
%%R
plotBCV(y)

In [ ]:
%%R
colnames(y$design)


In [ ]:
%%R -o tt
myContrast <- makeContrasts("groupL.CD14_Mono-groupR.CD14_Mono", levels = y$design)
qlf <- glmQLFTest(fit, contrast=myContrast)
# get all of the DE genes and calculate Benjamini-Hochberg adjusted FDR
tt <- topTags(qlf, n = Inf)
tt <- tt$table

In [ ]:
tt[:5]


In [ ]:
adata_pb.obs['lib_size']=adata_pb.obs['lib_size'].astype(int)


In [ ]:
%%time
%%R -i adata_pb
outs <-fit_model(adata_pb)

In [ ]:
%%R
fit <- outs$fit
y <- outs$y

In [ ]:
%%R -i adata_pb -o de_per_cell_type
de_per_cell_type <- list()
for (cell_type in unique(colData(adata_pb)$cell_type)) {
    print(cell_type)
    # create contrast for this cell type
    myContrast <- makeContrasts(paste0("groupR.", cell_type, "-groupL.", cell_type), levels = y$design)
    # perform QLF test
    qlf <- glmQLFTest(fit, contrast=myContrast)
    # get all of the DE genes and calculate Benjamini-Hochberg adjusted FDR
    tt <- topTags(qlf, n = Inf)
    # save in the list with the results for all the cell types
    de_per_cell_type[[cell_type]] <- tt$table
}

In [ ]:
# get cell types that we ran the analysis for
cell_types = de_per_cell_type.keys()
# add the table to .uns for each cell type
for cell_type in cell_types:
    df = de_per_cell_type[cell_type]
    df["gene_symbol"] = df.index
    df["cell_type"] = cell_type
    sc_toolbox.tools.de_res_to_anndata(
        adata,
        df,
        groupby="cell_type",
        score_col="logCPM",
        pval_col="PValue",
        pval_adj_col="FDR",
        lfc_col="logFC",
        key_added="edgeR_" + cell_type,
    )
    df.to_csv(f"de_edgeR_{cell_type}.csv")

In [ ]:
sc.get.rank_genes_groups_df(adata, group="CD14_Mono", key="edgeR_CD14_Mono")[
    :5

]

In [ ]:
FDR = 0.01
LOG_FOLD_CHANGE = 1.5



def plot_heatmap(adata, group_key, group_name="cell_type", groupby="label"):
    cell_type = "_".join(group_key.split("_")[1:])
    res = sc.get.rank_genes_groups_df(adata, group=cell_type, key=group_key)
    res.index = res["names"].values
    res = res[
        (res["pvals_adj"] < FDR) & (abs(res["logfoldchanges"]) > LOG_FOLD_CHANGE)
    ].sort_values(by=["logfoldchanges"])
    print(f"Plotting {len(res)} genes...")
    markers = list(res.index)
    sc.pl.heatmap(
        adata[adata.obs[group_name] == cell_type].copy(),
        markers,
        groupby=groupby,
        swap_axes=True,
    )

def volcano_plot(adata, group_key, group_name="cell_type", groupby="label", title=None):
    cell_type = "_".join(group_key.split("_")[1:])
    result = sc.get.rank_genes_groups_df(adata, group=cell_type, key=group_key).copy()
    result["-logQ"] = -np.log(result["pvals"].astype("float"))
    lowqval_de = result.loc[abs(result["logfoldchanges"]) > LOG_FOLD_CHANGE]
    other_de = result.loc[abs(result["logfoldchanges"]) <= LOG_FOLD_CHANGE]

    fig, ax = plt.subplots()
    sns.regplot(
        x=other_de["logfoldchanges"],
        y=other_de["-logQ"],
        fit_reg=False,
        scatter_kws={"s": 6},
    )
    sns.regplot(
        x=lowqval_de["logfoldchanges"],
        y=lowqval_de["-logQ"],
        fit_reg=False,
        scatter_kws={"s": 6},
    )
    ax.set_xlabel("log2 FC")
    ax.set_ylabel("-log Q-value")

    if title is None:
        title = group_key.replace("_", " ")
    plt.title(title)
    plt.show()

In [ ]:
plot_heatmap(adata, "edgeR_CD14_Mono")


In [ ]:
volcano_plot(adata, "edgeR_CD14_Mono")


In [ ]:
from bioinfokit import analys, visuz


In [ ]:
df = sc.get.rank_genes_groups_df(adata, group="CD14_Mono", key="edgeR_CD14_Mono")
df["pvals"] = df["pvals"].astype("float")

df["-logQ"] = -np.log(df["pvals"])
visuz.GeneExpression.volcano(df=df, lfc='logfoldchanges', pv='pvals',show=True, color=( "#DB141C", "#D4D4D4","#0444B4"), geneid="names", genenames='deg', lfc_thr= [LOG_FOLD_CHANGE, LOG_FOLD_CHANGE],gstyle=2, sign_line=True , dim=(5,5))


In [ ]:
result = sc.get.rank_genes_groups_df(adata, group="CD14_Mono", key="edgeR_CD14_Mono").copy()
result["-logQ"] = -np.log(result["pvals"].astype("float"))
result

In [ ]:
adata.uns['log1p']['base'] = None

In [ ]:
?sc.get.rank_genes_groups_df

In [ ]:
?dc.plot_volcano

In [ ]:
logFCs, pvals = dc.get_contrast(
    adata,
    group_col="cell_type",
    condition_col="side",
    condition="L",
    reference="R",
    method="wilcoxon",
)

In [ ]:
import decoupler as dc
dc.plot_volcano(logFCs, pvals, "CD14_Mono", top=5, sign_thr=0.05, lFCs_thr=1.5)


In [ ]:
np.log(0.582187)

In [ ]:
import pertpy as pt

In [ ]:
sccoda_model = pt.tl.Sccoda()

In [ ]:
adata

In [ ]:
sccoda_data = sccoda_model.load(
    adata,
    type="cell_level",
    generate_sample_level=True,
    cell_type_identifier="cell_type",
    sample_identifier="sample",
    covariate_obs=["side"],
)
sccoda_data

In [ ]:
pt.pl.coda.boxplots(
    sccoda_data,
    modality_key="coda",
    feature_name="side",
    figsize=(12, 5),
    add_dots=True,
    args_swarmplot={"palette": ["red"]},
)
plt.show()

In [ ]:
pt.pl.coda.stacked_barplot(
    sccoda_data, modality_key="coda", feature_name="side", figsize=(4, 2)
)
plt.show()

In [ ]:
sccoda_data = sccoda_model.prepare(
    sccoda_data,
    modality_key="coda",
    formula="side",
    reference_cell_type="automatic",
)
sccoda_model.run_nuts(sccoda_data, modality_key="coda", rng_key=1234)

In [ ]:
sccoda_data["coda"].varm

In [ ]:
sccoda_data["coda"].varm['effect_df_side[T.R]']


In [ ]:
sccoda_model.set_fdr(sccoda_data, 0.2)


In [ ]:
sccoda_model.credible_effects(sccoda_data, modality_key="coda")


In [ ]:
pt.pl.coda.effects_barplot(sccoda_data, "coda", "side")
plt.show()